# openpilot supercombo on comma2k19: change the image, see the plan

> **Outputs are not committed: run the notebook to see them.** Every figure, table and number the text refers to comes from a run, and git keeps none of them: `tests/test_notebooks_are_bare.py` holds this file to that.
>
> 1. **Once:** `make setup`, then `uv pip install --python .venv/bin/python -e ".[analysis]"` for Jupyter and what the notebook draws with.
> 2. **The site:** `paddock.toml` (or `PADDOCK_CONFIG`) says where `{runs}` and the data are. It reads run `20260916T051616-2a91ca`'s records (`RUN_ID`, §1) under `{runs}`, and the comma2k19 segments, each decoded from its start to the keyframe.
> 3. **Run it:** run all on the repo's `.venv` kernel, or headless from the repo root: `.venv/bin/jupyter execute --output='{notebook_name}_all_executed' tools/models/openpilot/analysis/openpilot_drv_supcombo_comma2k19_image_tweak.ipynb`. That writes an executed copy beside this file, which git ignores. Never `--inplace`: it writes the outputs into this file, and `make check` fails on that.
>
> **What it needs.** openpilot's checkout at the sha `repos.yaml` pins, with its own environment set up (`paddock doctor` says whether it is), because the notebook starts image nodes ([`openpilot_drv_supcombo_image_node.py`](openpilot_drv_supcombo_image_node.py), which must stay beside it) in openpilot's interpreter: one per turn, up to five, on CPU by default. Results and decoded frames are cached under `{runs}/comma2k19/_analysis/openpilot/image_tweak/`, so only a setting's first run costs model time.

This is the comma2k19 twin of [the KITScenes image-tweak notebook](openpilot_drv_supcombo_kitscenes_image_tweak.ipynb), and a companion to [the comma2k19 open-loop notebook](openpilot_drv_supcombo_comma2k19_openloop_analysis.ipynb). It runs no sweeps. You edit one config and see what it does:

1. **Five turns, as stock plans them.** For each turn:
    - the four images the model was given (each view's two frames);
    - the GT and the plan drawn on the keyframe;
    - the warps that cut each view out of the camera image, and the road's homography into each view;
    - the camera, and the rows openpilot expects the horizon on.
2. **One turn, `FOCUS`: how the height and speeds are measured.** Both the real values and the values the model reads.
3. **`CONFIG`, a change to what the model is fed, on that turn.** Edit it and re-run the cell. The model plans again, and the new plan is drawn beside stock's (or beside any other config). A `noise` row shows how much a half-pixel shift alone changes each number. A config can change:
    - the image: its geometry, sampling and tone;
    - the source: a cut of the frame to a narrower field of view;
    - the intent: whether the route's turn is sent as openpilot's desire.
4. **The same config on all five turns**, scored against stock (or any other config) on ADE, alignment with the GT and the speed profile. Each change is judged against a noise floor.
5. **Several configs side by side on all five turns**, for example stock, with intent, and with a lane-change desire as its control.

**What is different from KITScenes.** comma2k19 was recorded on comma's own camera, the EON's, which is what supercombo was trained on:
- There is one camera, and its frame is already comma's 65° cut, so `camera` is not a knob here and `fov_deg` can only cut narrower.
- The camera's height is comma's mount, 1.22 m, and each route has its own calibration (the mount's roll, pitch and yaw).
- The frames come at 20 Hz, openpilot's own rate, so every frame is one model tick and img's older frame is 5 frames back.
- **The GT is the device's path, not the calibrated frame the plan is in.** The harness scores the plan, which openpilot gives in its calibrated frame, against the device's own path. So the route's mount yaw, a degree or two, is part of every score. Part 2 shows what it is worth on `FOCUS`.

On every view the model is shown, a solid white line marks the row openpilot expects the horizon on: row 47.6 of img's 256, row 151.8 of big_img's.

**How it runs**
- **The code** is in [`openpilot_drv_supcombo_comma2k19_image_tweak_lib.py`](openpilot_drv_supcombo_comma2k19_image_tweak_lib.py), next to this notebook. Restart the kernel after editing it.
- **Paddock never imports a model, and neither does this notebook.**
    - The model runs in [`openpilot_drv_supcombo_image_node.py`](openpilot_drv_supcombo_image_node.py), the node the KITScenes notebooks use. It is the stock adapter; only the way it makes the warps and tensors is changed.
    - The node runs in openpilot's own interpreter, the same way `paddock run` launches a model. The notebook sends it matrices and lookup tables over localhost.
    - The checks before part 3 show that, with no config, the node is the stock adapter bit for bit, and that it reproduces the recorded run.
- **It runs on CPU by default.** The nodes start the first time they are needed, one per turn.
- **Every result is cached** under `{runs}/comma2k19/_analysis/openpilot/image_tweak/`. The cache key is the matrices sent, the sample, the cut it came from, the desire, the device and a hash of the code. Running the same config again costs nothing.
- **The frames** are decoded once from each segment's video, as the harness decodes them, and kept as raw arrays under the same directory. A cut (`fov_deg`) is staged once per turn beside them.

**As committed, it runs stock.** `CONFIG` leaves the image unchanged and part 5 has the one arm, stock, so a run shows stock against itself in parts 3–5. The top of this notebook says how to run it.

In [ ]:
import sys

import pandas as pd

from paddock.config import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tools/models/openpilot/analysis"))  # the lib, beside this notebook
import openpilot_drv_supcombo_comma2k19_image_tweak_lib as T

pd.set_option("display.max_columns", 50)  # part 4's per-turn table has 31 columns

## Parameters

- **`SCENES`**: the five turns. Wherever a cell takes a key, the key can be:
    - a sample id, such as `"comma2k19/9/99c94dc769b5d96e|2018-11-03--23-08-44/8/159"` (the chunk, the route, the segment, then the keyframe);
    - a label such as `"11-03--23-08-44/8/159"` (the route's time, the segment, then the keyframe);
    - a unique prefix of either.
- **`FOCUS`**: the turn that parts 2 and 3 work on.
- **`RUN_ID`**: the recorded stock run. Its samples are the ones a key is searched among, and the checks hold stock here to it.
- **`DEVICE`**: `"cpu"`, or `"cuda"` if a GPU is free. **`NODES`** is the number of model processes; **`THREADS`** is the threads each one uses.
- **`HEIGHT`**: the camera's real height. It is used to draw the road and by the knobs that move the camera. One of:
    - `"harness"`: the harness's, comma's mount, 1.22 m;
    - a number in metres.
- **`OVERRIDES`**: tag parameters for the turn classes, for example `{"curv_under": 0.6}`.
- **`RECOMPUTE`**: ignore the cached results.
- **`OVERLAY`**: where the GT and the plans are drawn.
    - `"img"`: the keyframe of the narrow view.
    - Add `"big_img"` for the wide view, or `"camera"` for the photo.

In [ ]:
SCENES = [
    "comma2k19/6/99c94dc769b5d96e|2018-07-11--15-13-12/14/789",  # left at 6.7 m/s: understeer; the worst ADE
    "comma2k19/5/99c94dc769b5d96e|2018-07-05--16-30-16/13/999",  # right at 11.3 m/s: understeer
    "comma2k19/9/99c94dc769b5d96e|2018-11-03--23-08-44/8/159",  # right at 5.7 m/s, at night: not planned
    "comma2k19/3/99c94dc769b5d96e|2018-05-03--09-25-46/36/579",  # left at 6.2 m/s: oversteer; the best ADE
    "comma2k19/1/b0c9d2329ad1606b|2018-07-29--12-02-42/30/369",  # right at 4.9 m/s: matched (a control)
]
FOCUS = "11-03--23-08-44/8/159"
RUN_ID = "20260916T051616-2a91ca"  # the recorded comma2k19 stock run
DEVICE = "cpu"
NODES, THREADS = 5, 8
HEIGHT = "harness"
OVERRIDES = {}
RECOMPUTE = False
OVERLAY = ("img",)  # add "big_img" or "camera" to see more of the path

lab = T.Lab(
    SCENES,
    RUN_ID,
    device=DEVICE,
    nodes=NODES,
    threads=THREADS,
    height=HEIGHT,
    overrides=OVERRIDES,
    recompute=RECOMPUTE,
)

## 1 Five turns, as stock plans them

"Stock" here is stock run by the node, on this machine. It is the recorded run's model and samples: check 3 below shows it reproduces that run's plans to within 0.03 m over 3 s on every turn. It is used rather than the recorded plans because the recorded run predates two of the extras the tables read, `plan_v0` and the desire state.

The five turns cover the four turn classes that stock shows on them:

| turn | route | speed | GT's turn over 3 s | stock's | stock |
|---|---|---|---|---|---|
| 07-11--15-13-12/14/789 | left | 6.7 m/s | 86° | 42° | turns half as far by 3 s; the worst ADE |
| 07-05--16-30-16/13/999 | right | 11.3 m/s | 22° | 14° | turns too little |
| 11-03--23-08-44/8/159 | right, at night | 5.7 m/s | 36° | −4° | does not plan the turn |
| 05-03--09-25-46/36/579 | left | 6.2 m/s | 31° | 42° | turns too far; the best ADE |
| 07-29--12-02-42/30/369 | right | 4.9 m/s | 64° | 47° | turns 0.73 of the GT's turn, which the classes call matched |

The columns, here and in every table below:
- **`mode`**: the open-loop notebook's turn class. The first match wins:
    - `not_planned`: the plan's heading changes by less than 15° over its 10 s;
    - `wrong_way`: it turns 15° or more away from the route;
    - otherwise the class compares the plan's turn with the GT's over 3 s: `understeer` under 0.7 of it, `oversteer` 1.3 or more, `matched` in between.
- **`ade`**: the benchmark's score, the mean distance from the GT over 3 s (m).
- **`ade_long`**: the same over the GT's whole length, up to 10 s.
- **`lat`**: the benchmark's mean lateral error over 3 s (m), between the plan and the GT at the same time.
- **`path_err`**: the mean distance from the plan's waypoints over 3 s to the GT's path, wherever along it they fall (m). It ignores timing: a plan on the GT's line scores 0, however slowly it drives it.
- **`dpsi`, `gt_dpsi`**: the plan's and the GT's heading change over 3 s (deg). Positive is toward the route.
- **`turn_frac`**: the plan's share of the GT's turn, over the GT's whole length.
- **`speed_rmse`**: the RMS difference between the plan's and the GT's speeds over 3 s, per 0.5 s step (m/s).
- **`v0_ratio`**: the speed of the plan's first step over the GT's.
- **`head_ratio`**: the pose head's speed over the car's.
- **`plan_v0`**: the plan's own start speed (m/s).
- **`cam_h`**: the camera height the model reads (m). The real one is comma's mount, 1.22 m. Part 2 says how each is measured.
- **`p_turn`** (parts 3–5): the probability the model's desire-state head gives the route's turn (turn left or turn right) at the keyframe. It shows whether the model thinks it is turning, whether or not it was sent the intent.

In [ ]:
lab.stock_table().round(3)

Each figure shows one turn:
- **Top row:**
    - the camera's keyframe, with the outlines of img (yellow) and big_img (cyan) and the road at its real height;
    - the paths seen from above;
    - the speed and the heading at each 0.5 s step.
- **Bottom row:** the four images the model was given. Each view has two frames: the frame 0.2 s before the keyframe, then the keyframe. The GT and the plan are drawn on the keyframe of each `OVERLAY` view, placed where the real camera sees them.

In [ ]:
for key in lab.keys:
    lab.show_stock(key, overlay=OVERLAY)

**How much of the path img sees.** Each view's title counts the waypoints inside it. img spans ±16° across and about 3° above to 13° below the camera's axis. On four of the five turns, it shows at most 2 of the GT's 20 waypoints:
- the near part of the path lies below img's bottom edge;
- after that, the turn takes the path out of img's sides.

Only 07-05--16-30-16/13/999 keeps part of its path inside img: 6 of its 20 GT waypoints. That turn is at 11.3 m/s on a gentle bend. On `FOCUS` the plan goes straight, so 19 of its 20 waypoints are in img while only 2 of the GT's are.

To draw the path where it can be seen, set `OVERLAY = ("img", "big_img", "camera")`. big_img spans ±29° across, and the photo shows the camera's whole field of view.

### The warp and the homographies

The model never sees the camera's image directly. Each view is cut out of it by a 3 × 3 homography, one per view for each route:

`W[view] = camera_from_calib @ calib_from_model[view]`

- **`calib_from_model[view]`** is openpilot's own and is the same for every car. It maps a model pixel to a ray in the calibrated frame (x forward, y right, z down).
- **`camera_from_calib`** belongs to the route: the EON's intrinsics K, times the mount's rotation (the roll, pitch and yaw in the route's calibration).

So W contains a rotation and a focal change, but no camera height. The height enters through the road's homography into each view:

`inv(W[view]) @ G`, where `G = K @ [r1 r2 t]`

G maps a road point (x, y, 1) to a camera pixel, for a camera h metres above the road.

The cell below shows three things:
1. the matrices for `FOCUS`;
2. the road grid drawn through each view's homography, on all five turns;
3. a table of each turn's camera.

Part 3's image changes are applied between the two factors of W:

`camera_from_calib @ calib @ calib_from_model[view]`

In [ ]:
lab.show_warps(FOCUS)

### The camera, and the horizon openpilot expects

On KITScenes a config could feed the model either of two front cameras, cut to comma's 65°. Here there is nothing to choose: comma2k19 was recorded on the EON's road camera, 1164 × 874 px at a focal length of 910 px, 65.2° across. openpilot defines img at that focal length, so one img pixel is one camera pixel; big_img spans about two.

`fov_deg` can still cut the frame narrower, about its centre, and by default resamples the cut back to 1164 px across. The model is told the cut's true focal length, so its views still look at the same scene, and check 5 below confirms it for a 50° cut. Under about 58° (big_img spans ±29°), big_img reaches past the cut's edge.

The figure shows `FOCUS`: the keyframe with its views' footprints, then img and big_img at the keyframe. On each view:
- **solid white**: the row openpilot expects the horizon on. It is the row of the calibrated straight-ahead ray in openpilot's `calib_from_model`, 47.6 of 256 in img and 151.8 of 256 in big_img, the same for every car;
- **dashed white**: this camera's own horizon, through the route's calibration. With openpilot's warp the two coincide by construction, so here the dashed line hides under the solid one. In part 3, a `pitch_deg`, `roll_deg` or `fov_told_deg` moves it off.

The table gives the camera's frame, focal length, field of view, height, mounting angles, where its calibration comes from, and the camera pixels per model pixel in each view.

In [ ]:
lab.show_cameras(FOCUS)

## 2 How the height and speeds are measured, on one turn

`FOCUS`, 11-03--23-08-44/8/159, is a night turn on a wet road, with glare and brake lights ahead. It has the worst ADE but one of the five, and its plan misses the turn:
- The GT turns 36° right over 3 s and slows from 5.3 to 2.6 m/s. The plan turns 3.5° the other way (`not_planned`); its `p_turn` is 0.013.
- The plan is also too fast. Its first step is 1.42× the GT's (7.6 m/s against 5.3), and it is still about twice the GT's speed at 3 s (the figure's bottom right).
- So most of its 5.1 m from the GT's path is the missed turn, and its speed error adds to the lateral error the benchmark sees.

Unlike KITScenes's `FOCUS`, the shape is not sensitive to the pixels here. Part 3's `noise` row shows that half-pixel shifts move its heading by 0.19° at most, its start speed by 2%, and its distance from the GT's path by 0.15 m.

The model reads its camera as high: 1.36 m, where comma's mount puts it at 1.22 m. Reading the camera too high and the car too fast go together, as below.

The model is told neither its height nor its speed. It reads both from the images, and two of its outputs show what it read:
- **`cam_h`**: the camera's height above the road plane, from supercombo's `road_transform` head;
- **the pose head**: the model's forward speed, from supercombo's `pose` head.

`plan_v0`, the plan's own start speed, is a separate output.

The tables below give every height and speed on this turn, with how each is measured. Nothing in comma2k19 measures the camera's height, so the harness's 1.22 m is comma's nominal mount, not a measurement. The figure shows how the two are linked:
- **Top row:** img's two frames, 0.2 s apart, and what changed between them. That change is all the image says about speed. At night, much of it is glare and lights, not road.
- **Bottom left:** the road 10–40 m ahead (green), and the image rows the model reads as those distances (red, dashed) at the height it reads. The plan is drawn twice: on the real road, and on the road the model reads.
    - **What geometry alone predicts** (argued). The model puts its camera at 1.36 m when it is 1.22 m up, so it reads every distance as 1.12 of the true one. It should therefore read the road between the frames as moving 1.12 as far, and so read 1.12× the true speed.
    - **What was measured.** The pose head reads 1.38× here, more than geometry predicts. On KITScenes the note found the opposite: the speed went as that ratio to the power 0.68, less than geometry. One turn does not say which holds on comma2k19.
- **Bottom right:** the speeds. The GT's and the plan's are plotted per 0.5 s step, with the readings at t = 0.

In [ ]:
lab.show_measurements(FOCUS)

## Before part 3: does the node give the stock model's output?

Five checks, each of which raises an error if it fails:
1. **The node is the stock adapter.** With no config, the node gives the same plan, extras, warps and ticks as the untouched adapter, bit for bit. So does an identity config sent the way every config is sent, as `calib` and `post` matrices. So a change in the plan is the config's own, and the half-pixel shifts behind the `noise` row and part 4's floor measure the shift alone.
2. **The drawings show what the model was fed.** What this notebook draws as the model's input matches what the model received. This is checked on every path the node has: shift, supersampling, filter, tone tables and blanking.
3. **The samples are the recorded run's, and stock here reproduces it.** The samples rebuilt here score the recorded plans exactly as the recorded run scored them, so they are the samples that run was given. Stock run here matches the recorded plans within 0.25 m over 3 s on every turn. The recorded run was made on a GPU, so the match cannot be bit for bit.
4. **The tick counts agree.** The model ticks counted here are the node's own. img's older frame is found from them.
5. **A cut does not move the views.** For a 50° cut (`T.CHECK_CUT_DEG`), the warp openpilot computes from the cut's intrinsics equals the stock warp followed by the map from camera pixels to cut pixels, to within 0.01 cut px over a grid on each view. The mounting angles openpilot derives are unchanged. So a cut changes which pixels the views are sampled from, not where they look.

In [ ]:
lab.checks(FOCUS)

## 3 Change the image or the intent, on one turn

Edit `CONFIG` and run the cell. The model plans `FOCUS` again with the image changed, and the figure shows the result beside stock:
- **Top row:**
    - the keyframe, with stock's view outlines (grey) and the config's;
    - the road grid as the config shows it to the model (red), against where it really is (grey);
    - the paths seen from above.
- **Middle row:** the speed and the heading at each 0.5 s step.
- **Bottom rows:** the four images the model is given, stock's above the config's.
    - On the `OVERLAY` views, the GT and the plan are drawn.
    - Two horizons are drawn on each keyframe view: the row openpilot expects the horizon on (solid white, labelled: row 47.6 of img's 256, 151.8 of big_img's), and where the camera's own horizon lands in the view the config makes (dashed). Stock puts them together; a config that pitches, rolls or mis-tells the field of view pulls them apart.

The table under the figure gives part 1's metrics for stock, for the config, and the change between them. Its last row, **`noise`**, is the largest change that any of eight half-pixel shifts (`T.SHIFTS`) makes on this turn, with every turn class they give.
- The shifts move the geometry by almost nothing, so a change within `noise` is not the config's doing.
- They take eight runs the first time on each turn, cached after that. `lab.tweak(..., noise=False)` skips them.

**How to use it**
- Every value in `CONFIG` below leaves the image unchanged, and the knobs combine freely.
- Set **`RUN_MODEL = False`** to draw only what the model would be shown, in a second or two, without running the model.
- To run another turn, use `lab.tweak(key, CONFIG)` with any key.
- **`AGAINST`** sets the baseline the config is drawn and scored beside: `None` is stock, or give another config. For example, to see what intent does on a 50° cut, set `CONFIG = dict(fov_deg=50, desire="turn")` and `AGAINST = dict(fov_deg=50)`. The `noise` row then shifts the baseline, not stock.

| knob | what the model is shown | what it should do to the plan |
|---|---|---|
| `height_scale` s | the road as seen from a camera at h / s | every distance and width s× larger, so faster |
| `zoom` k | the image magnified k×, like a longer lens | distances ×1/k with widths unchanged, so slower when k > 1; a smaller field of view |
| `hstretch` g | the image stretched g× sideways | lateral offsets and lane widths ×g; curves look tighter |
| `yaw_deg` ψ | the view turned ψ left | the plan's heading turns ψ right |
| `turn_yaw_deg` | the view turned toward the route's side; negative turns it away | turned toward: the plan turns away from the route; turned away: the plan turns toward it |
| `pitch_deg` | the view tilted up, so the road sits lower | the road reads nearer, so slower |
| `roll_deg` | the view rolled, so the right side looks lower | the plan drifts sideways |
| `ahead_m`, `left_m` | the camera moved over the road, at the same height | the plan starts where that camera would stand |
| `dx_px`, `dy_px` | the sampling grid moved right and down, in model pixels | nothing beyond noise; half a pixel is the noise floor |
| `views` | which views the geometry applies to: `"both"`, `"img"` or `"big_img"` | |
| `supersample` n, `ss_views` | each model pixel the mean of n × n camera samples, in `ss_views` | big_img spans about 2 camera pixels per model pixel here, as on comma's camera everywhere: little to average |
| `sharpen` α | an unsharp mask, δ + α(δ − blur) over 3 × 3; −1 is the blur itself | |
| `gamma`, `contrast`, `brightness`, `saturation` | tone, on both views: gamma on luma (below 1 is brighter); contrast about mid-grey; brightness in luma steps; saturation 0 is grey | |
| `blank` | `"img"` or `"big_img"` shown as flat grey | shows what that view contributes |
| **the source** | | |
| `fov_deg` θ | the frame cut to θ° across about its centre before openpilot sees it; 0 is the frame as recorded, which is already the EON's 65.2° | the views still look at the same scene, sampled through the cut's pixels. Under about 58° (big_img spans ±29°), big_img reaches past the cut's edge. Once the model has run, the line under the table gives each view's share inside the frame it was fed |
| `fov_aspect` | the cut's width over its height; the EON's 1164 / 874 by default | a cut too tall for the frame raises an error that says so |
| `fov_width` w | the cut resampled to w px across; the EON's 1164 by default, so a narrower cut is magnified back to the frame's width. 0 crops whole camera pixels and resamples nothing | how finely the cut is sampled before openpilot warps it |
| `fov_filter` | the resampling filter: `"area"` (the box mean, as a sensor bins), `"bilinear"`, `"lanczos"` or `"nearest"` | |
| `fov_told_deg` | the model told the cut is this many degrees across, not its true width; 0 tells it the truth. Without a cut, nothing to tell | openpilot computes its warp from the wrong focal length, so the views are magnified by about tan(told / 2) / tan(θ / 2) about the centre, as `zoom` does; told wider is a longer lens |
| **the intent** | | |
| `desire` | `"off"` (stock: every frame sent STRAIGHT, as the recorded run sent); `"turn"`: the route's side as openpilot's `turnLeft` / `turnRight`; `"lane_change"`: as `laneChangeLeft` / `laneChangeRight`, in distribution but a different manoeuvre, so a control for `"turn"` | `p_turn` shows whether the model takes it up |
| `desire_lead_s` | how long before the keyframe the desire starts, 0 to 7.95 s (the sample's 160 frames) | openpilot pulses a desire once, on its first frame, and its queue holds it for 5 s. Up to 5 s the model sees it at the keyframe, and the lead sets how long it has had it; past 5 s it has dropped out of the queue by the keyframe, which makes a control |

**Where to start.**
- **This camera is the one supercombo was trained on.** comma2k19 is comma's own EON at comma's mount, so stock is in distribution, and the frame needs no cut. The KITScenes note's `height_scale` numbers (s ≈ 1.2) were measured on a camera 1.87 m up, and do not transfer: here they would make the plan faster still.
- **Speed.** Where KITScenes's plans start too slow, three of these five start too fast: `v0_ratio` 1.30, 1.42 and 1.20. On `FOCUS` the model reads its camera high (1.36 m against 1.22) and the car fast (`head_ratio` 1.38). A `height_scale` below 1 shows it the road from a higher camera, so every distance reads shorter; whether its speed follows by geometry alone (s), or less (as on KITScenes), is what a sweep would measure.
- **Turning.** `FOCUS` does not plan the turn at all, and its `p_turn` is 0.013, so the model sees no turn coming. `desire="turn"` is the natural first arm here. `desire="lane_change"` is its control, and `desire_lead_s` past 5 s is a second one.
- **This turn's shape is not sensitive to the pixels.** All eight shifts move its heading by under 0.2°, so a config that turns this plan a few degrees is doing more than any change does.

In [ ]:
CONFIG = dict(
    # geometry
    height_scale=1.0,  # s: the road as from a camera at h / s
    zoom=1.0,  # k: magnified k x
    hstretch=1.0,  # g: stretched g x sideways
    yaw_deg=0.0,  # the view turned left (deg)
    turn_yaw_deg=0.0,  # the view turned toward the route's side (deg); negative is away
    pitch_deg=0.0,  # the view tilted up (deg)
    roll_deg=0.0,  # the view rolled, its right side down (deg)
    ahead_m=0.0,  # the camera moved forward (m)
    left_m=0.0,  # and left (m)
    dx_px=0.0,  # the sampling grid moved right (model pixels)
    dy_px=0.0,  # and down
    views="both",  # where the geometry applies: "both", "img" or "big_img"
    # sampling
    supersample=1,  # n: n x n camera samples per model pixel
    ss_views="big_img",  # where supersampling applies
    sharpen=0.0,  # alpha; -1 is a 3 x 3 blur
    # tone, on both views
    gamma=1.0,  # on luma; under 1 is brighter
    contrast=1.0,  # about mid-grey
    brightness=0.0,  # luma steps
    saturation=1.0,  # 0 is grey
    # ablation
    blank="",  # "img" or "big_img": that view shown as flat grey
    # the source
    fov_deg=0.0,  # cut the frame to this many degrees across, under the EON's 65.2; 0 is as recorded
    fov_aspect=T.EON_W / T.EON_H,  # the cut's width over height
    fov_width=T.EON_W,  # resample the cut to this many px across; 0 crops whole pixels
    fov_filter="area",  # "area", "bilinear", "lanczos" or "nearest"
    fov_told_deg=0.0,  # tell the model the cut is this wide (deg); 0 tells it the truth
    # the intent
    desire="off",  # "off", "turn" or "lane_change"
    desire_lead_s=0.0,  # the desire starts this long before the keyframe (s), 0 to 7.95
)
AGAINST = None  # the baseline: None is stock, or a config such as dict(fov_deg=50)
RUN_MODEL = True  # False: draw what the model would be shown, without running it

lab.tweak(FOCUS, CONFIG, run=RUN_MODEL, overlay=OVERLAY, against=AGAINST)

## 4 The config on all five turns

Run this once part 3's config looks worth scoring. It plans the five turns twice, with `CONFIG` and with part 3's `AGAINST` (stock when it is `None`), both here. It shows:

1. **Each turn**, stock and config side by side, with the change. The columns are part 1's. `dpsi_err` is the plan's heading change over 3 s minus the GT's; it is negative when the plan turns too little.
2. **The means over the turns, with a verdict for each criterion.** Lower is better for every criterion except the count of plans that turn toward the route:

    | group | criteria |
    |---|---|
    | ADE | ADE over 3 s (the benchmark's score); ADE over the GT's whole length, up to 10 s |
    | alignment | lateral error over 3 s; distance from the GT's path over 3 s; heading error over 3 s; how many plans turn toward the route (higher is better) |
    | speed profile | RMSE of the 0.5 s step speeds over 3 s; how far the plan's start speed is from the GT's; how far the pose head's speed is from the pose speed |


    Two rows follow without a verdict: how many plans turn toward the route, and the mean `p_turn`.
3. **One verdict per group**: better, worse, mixed, or no clear change.
4. **Each turn drawn**: the paths, the speeds and the headings. Stock is orange; the config is purple.

**Read alignment with the speed profile.** The lateral and heading errors compare the plan and the GT at the same time, as the benchmark does, so a plan that is only too slow on a curve shows both. The distance from the GT's path ignores timing. In part 1's table:
- 07-05--16-30-16/13/999 keeps to the GT's path (`path_err` 0.56 m) but slows from 10.2 to about 6 m/s while the car speeds up from 11.3 to 18.8 m/s over 10 s. So its lateral error over 3 s is 2.00 m, and its heading over 3 s is 14° of the GT's 22°: understeer by the clock, though not by the road.
- `FOCUS` is off the GT's path (`path_err` 5.06 m) because it misses the turn, and fast, so both show.

**The noise floor.** Any change at all to the image moves every score a little. So a change counts only when it is bigger than what eight shifts of half a model pixel do (`T.SHIFTS`); those shifts move the geometry by almost nothing.
- **Each turn:** a turn counts as better or worse when its change exceeds the median change a shift makes on one turn.
- **The mean:** the verdict is better when the mean change exceeds the largest change any shift makes to the mean, and more turns get better than worse. Worse is the same in reverse.
- **Each group:** better when one of its criteria is better and none is worse; mixed when some are better and some worse.
- `lab.compare(CONFIG, noise=False)` counts any change at all.

The first call runs the shifts on the five turns: 40 runs, cached after that. With `AGAINST` set, the shifts are of that baseline, so they run again for each new baseline.

Five turns is a small sample. A clear verdict here is a reason to test on more turns, not a result. `SCENES` accepts any keyframe of the recorded run.

In [ ]:
lab.compare(CONFIG, against=AGAINST)

## 5 Several configs side by side

`ARMS` names up to five configs, each written like `CONFIG` (only the knobs it changes). The cell plans all five turns with each, and shows:

1. **The arms**, each with the knobs it changes. In the figures each arm keeps the colour of its place in the list: the first is orange, the second purple, then pink, green and lilac. Adding an arm at the end leaves the others' colours as they were.
2. **Each turn**, every arm's metrics side by side.
3. **The means over the turns.** Each later arm is judged against the first, with part 4's noise floor for the first arm: on each criterion, a verdict and how many turns got better, stayed within the floor, and got worse. Then, without verdicts, the count of plans that turn toward the route and the mean `p_turn`.
4. **Each turn drawn**: the paths, the speeds and the headings, one colour per arm.

**Put the baseline first.** To ask what intent does, compare stock and stock with `desire="turn"`. The commented arms below add intent, its lane-change control, intent that has dropped out of openpilot's queue by the keyframe, and a 50° cut.

As committed it has only the one arm, stock, so a run shows stock's own numbers and no verdicts.

In [ ]:
ARMS = {
    "stock": {},
    # "intent": dict(desire="turn"),
    # "lane-change desire": dict(desire="lane_change"),  # the control for "intent"
    # "intent, 6 s early": dict(desire="turn", desire_lead_s=6.0),  # out of openpilot's 5 s queue by the keyframe
    # "50 deg cut": dict(fov_deg=50.0),
    # "config": CONFIG,
}

lab.arms(ARMS)

## Stopping the nodes

The nodes also stop when the kernel stops, and start again on the next call that needs one.

In [ ]:
lab.close()